Extract Unused Mathdial Dataset

In [47]:
import pandas as pd
import json
import numpy as np

In [48]:
meta_data = json.load(open("./data/meta_data/stepverify_0.9__student_converation_match.json"))

In [49]:
# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.jsonl', 'test': 'test.jsonl'}
mathdial_train_df = pd.read_json("hf://datasets/eth-nlped/mathdial/" + splits["train"], lines=True)
mathdial_test_df = pd.read_json("hf://datasets/eth-nlped/mathdial/" + splits["test"], lines=True)

# concat train and test sets
mathdial_df = pd.concat([mathdial_train_df, mathdial_test_df], ignore_index=True)

In [50]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import normalize_text
from typing import Any
import re

def parse_mathdial_conversation(conversation: Any) -> list[dict[str, Any]]:
    turns: list[dict[str, Any]] = []

    for raw_turn in conversation.split("|EOM|"):
        turn = raw_turn.strip()
        if not turn or ":" not in turn:
            continue

        speaker, text = turn.split(":", 1) # split the turn into speaker and text ( max 2 items)
        speaker_key = speaker.strip().lower()
        text = text.strip()
        label = None

        if speaker_key == "teacher":
            label_match = re.match(r"^\s*\(([^()]+)\)\s*(.*)$", text, flags=re.DOTALL) # (..)text, include dots

            if label_match:
                label = label_match.group(1).strip().lower()
                text = label_match.group(2).strip()
            else:
                label = None
                text = text

            user = "teacher"

        else:
            user = "student"
            label = None
            text = text

        turns.append(
            {
                # "turn_idx": len(turns),
                "user": normalize_text(user),
                # "label": normalize_text(label),
                # "text": text,
                "text": normalize_text(text),
            }
        )

    return turns



In [51]:
def get_unused_mathdial_rows(mathdial_df, meta_data):
    mathdial_df_row_idx = mathdial_df.reset_index(drop=True).copy()
    mathdial_df_row_idx.insert(0, "mathdial_row_id", np.arange(len(mathdial_df_row_idx))) # mathdial dataset with row ids ( the same way how mathdial was assigned indices when labeling stepverify data with mathdial )

    used_mathdial_row_ids = list(set([row["mathdial_row_id"] for row in meta_data]))
    mathdial_unused_rows = mathdial_df_row_idx[~mathdial_df_row_idx["mathdial_row_id"].isin(used_mathdial_row_ids)]
    
    print(f"Total mathdial rows: {len(mathdial_df)}")
    print(f"Used mathdial rows: {len(used_mathdial_row_ids)}")
    print(f"Unused mathdial rows: {len(mathdial_unused_rows)}")

    return mathdial_unused_rows


In [52]:
def get_unused_mathdial_rows_with_parsed_conversation(mathdial_df, meta_data):
    mathdial_unused_rows = get_unused_mathdial_rows(mathdial_df, meta_data)
    mathdial_unused_rows['parsed_conversation']= mathdial_unused_rows['conversation'].apply(parse_mathdial_conversation)
    return mathdial_unused_rows

In [53]:
mathdial_unused_rows_df = get_unused_mathdial_rows_with_parsed_conversation(mathdial_df, meta_data)

Total mathdial rows: 2861
Used mathdial rows: 999
Unused mathdial rows: 1862


#### Get student teacher pairs 

In [54]:
# first_turn_users = [row['parsed_conversation'][0]['user'] for row in mathdial_unused_rows_dict]
# set(first_turn_users)

In [55]:
import ast
import pandas as pd

# question, 

def make_student_teacher_pairs(mathdial_unused_rows_dict):
    pair_rows = []

    for row in mathdial_unused_rows_dict:
        context_data_idx = row["mathdial_row_id"]
        conversation = row["parsed_conversation"]
        qid = row["qid"]
        scenario = row["scenario"]
        problem = row["question"]
        ground_truth = row["ground_truth"]
        student_profile = row["student_profile"]
        self_correctness = row["self-correctness"]
        teacher_described_confusion = row["teacher_described_confusion"]
        self_typical_confusion = row["self-typical-confusion"]
        self_typical_interactions = row["self-typical-interactions"]
        
        # Exclude only the first teacher turn
        if conversation[0].get("user") == "teacher":
            conversation = conversation[1:]

        pair_idx = 0

        # student 바로 다음에 teacher가 있는 경우만 pair 생성
        for i in range(len(conversation) - 1):
            current_turn = conversation[i]
            next_turn = conversation[i + 1]

            if (current_turn.get("user") == "student") and (next_turn.get("user") == "teacher"):
                pair_rows.append({
                    "context_data_idx": context_data_idx,
                    "pair_idx": pair_idx,
                    "problem": problem, 
                    "student": current_turn["text"],
                    "teacher": next_turn["text"],

                    "ground_truth": ground_truth,
                    "student_profile": student_profile,
                    "self_correctness": self_correctness,
                    "teacher_described_confusion": teacher_described_confusion,
                    "self_typical_confusion": self_typical_confusion,
                    "self_typical_interactions": self_typical_interactions,
                    "qid": qid,
                    "scenario": scenario,
                })

                pair_idx += 1

    return pd.DataFrame(pair_rows)

In [59]:
import pandas as pd 

mathdial_unused_rows_dict = mathdial_unused_rows_df.to_dict(orient="records")
student_teacher_pairs_df = make_student_teacher_pairs(mathdial_unused_rows_dict)
student_teacher_pairs_df

,context_data_idx,pair_idx,problem,student,teacher,ground_truth,student_profile,self_correctness,teacher_described_confusion,self_typical_confusion,self_typical_interactions,qid,scenario
0,10,0,When the strawberries at Fruity Farm are ready...,sure. i let x be the number of pounds of straw...,well done for giving it a go but its not quite...,"To gain entry into the strawberry fields, Sall...",Michael is a 7th grade student. He struggle to...,Yes,they did not add the entrance fee on,4.0,3.0,5000064,1
1,10,1,When the strawberries at Fruity Farm are ready...,they paid 4 for entrance.,"and 3 went, so what would we do?","To gain entry into the strawberry fields, Sall...",Michael is a 7th grade student. He struggle to...,Yes,they did not add the entrance fee on,4.0,3.0,5000064,1
2,10,2,When the strawberries at Fruity Farm are ready...,"we would multiply the entrance fee by 3, since...",and what woud that equal?,"To gain entry into the strawberry fields, Sall...",Michael is a 7th grade student. He struggle to...,Yes,they did not add the entrance fee on,4.0,3.0,5000064,1
3,10,3,When the strawberries at Fruity Farm are ready...,"after subtracting the entrance fee, the equati...",if they all paid 4 dollars and there were 3 pe...,"To gain entry into the strawberry fields, Sall...",Michael is a 7th grade student. He struggle to...,Yes,they did not add the entrance fee on,4.0,3.0,5000064,1
4,10,4,When the strawberries at Fruity Farm are ready...,the total entrance fee would be 3 x 4 12.,great. now we know that they paid 128 for the ...,"To gain entry into the strawberry fields, Sall...",Michael is a 7th grade student. He struggle to...,Yes,they did not add the entrance fee on,4.0,3.0,5000064,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10134,2855,3,Taegan goes to a carnival where she wins ticke...,she won 5 tickets per game. 5 tickets x 5 game...,what is 5 divided by 5?,"If tickets are valued at $3 then in total, Tae...",Lakisha is a 7th grade student. She has proble...,Yes,Student didn't take the steps to solve the pro...,4.0,3.0,5000489,4
10135,2856,0,Tom's cat needs an expensive surgery. He has ...,"sure, i used the equation 24 x 20 480 to calcu...","okay, so tom was responsible for paying 1000 f...",The insurance cost 24*20=$480\nWith insurance ...,Luca is a 7th grade student. He has problem wi...,Yes,The student forgot to add the cost of insuranc...,4.0,4.0,5000286,5
10136,2856,1,Tom's cat needs an expensive surgery. He has ...,"right, so the total amount that tom paid for t...",tom saved 4000 because he had insurance. but h...,The insurance cost 24*20=$480\nWith insurance ...,Luca is a 7th grade student. He has problem wi...,Yes,The student forgot to add the cost of insuranc...,4.0,4.0,5000286,5
10137,2856,2,Tom's cat needs an expensive surgery. He has ...,tom paid 480 for the pet insurance over 24 mon...,to figure out the amount of money that tom sav...,The insurance cost 24*20=$480\nWith insurance ...,Luca is a 7th grade student. He has problem wi...,Yes,The student forgot to add the cost of insuranc...,4.0,4.0,5000286,5


In [60]:
import os

# Ensure the "sft_data" directory exists
os.makedirs("sft_data", exist_ok=True)

student_teacher_pairs_df.to_csv("sft_data/mathdial_unused_rows__student_teacher_pairs_df.csv")
mathdial_unused_rows_df.to_csv("sft_data/mathdial_unused_rows_df.csv")